# 주식선물 실시간 호가 수집기 (MM 헷징 분석용)
키움 Open API를 사용하여 주식선물 호가/시세 및 기초자산 체결 데이터를 CSV로 저장

## 1. 설정

In [1]:
# === 사용자 설정 ===
FUTURES_CODE = "A1164000" # 선물 종목코드 (예: 106S3000 - 삼성전자 선물) (26년 4월만기)
STOCK_CODE = "005930"   # 기초자산 종목코드 (삼성전자 주식)
SCREEN_NO = "1000"      # 화면번호

## 2. 라이브러리 Import

In [2]:
import sys
import os
from datetime import datetime
import pandas as pd
from PyQt5.QtWidgets import QApplication
from PyQt5.QAxContainer import QAxWidget
from PyQt5.QtCore import QEventLoop

## 3. CSV 저장 설정 및 FID 매핑

In [3]:
# 저장 디렉토리 설정
DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# CSV 파일 경로 생성
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
CSV_FILENAME = f"futures_{FUTURES_CODE}_{timestamp_str}.csv"
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

# CSV 컬럼 정의
CSV_COLUMNS = [
    # 기본 정보
    "timestamp", "data_type",
    
    # 선물 시세 (선물시세 Real Type)
    "futures_code", "futures_time", "futures_price", "futures_change", "futures_change_rate",
    "futures_volume", "futures_acc_volume", "futures_acc_amount",
    "futures_open", "futures_high", "futures_low",
    "futures_theoretical", "futures_theoretical_basis", "futures_market_basis",
    "futures_disparity_rate", "futures_disparity",
    "futures_open_interest", "futures_open_interest_change",
    
    # 선물 호가 (선물호가잔량 Real Type)
    "futures_ask_1", "futures_ask_qty_1", "futures_ask_cnt_1",
    "futures_ask_2", "futures_ask_qty_2", "futures_ask_cnt_2",
    "futures_ask_3", "futures_ask_qty_3", "futures_ask_cnt_3",
    "futures_ask_4", "futures_ask_qty_4", "futures_ask_cnt_4",
    "futures_ask_5", "futures_ask_qty_5", "futures_ask_cnt_5",
    "futures_bid_1", "futures_bid_qty_1", "futures_bid_cnt_1",
    "futures_bid_2", "futures_bid_qty_2", "futures_bid_cnt_2",
    "futures_bid_3", "futures_bid_qty_3", "futures_bid_cnt_3",
    "futures_bid_4", "futures_bid_qty_4", "futures_bid_cnt_4",
    "futures_bid_5", "futures_bid_qty_5", "futures_bid_cnt_5",
    "futures_ask_total", "futures_bid_total",
    
    # 기초자산 (주식체결 Real Type)
    "stock_code", "stock_time", "stock_price", "stock_change", "stock_change_rate",
    "stock_volume", "stock_acc_volume",
    "stock_open", "stock_high", "stock_low"
]

# FID 매핑 - 선물시세
FID_FUTURES_TRADE = {
    "futures_time": 20,
    "futures_price": 10,
    "futures_change": 11,
    "futures_change_rate": 12,
    "futures_volume": 15,
    "futures_acc_volume": 13,
    "futures_acc_amount": 14,
    "futures_open": 16,
    "futures_high": 17,
    "futures_low": 18,
    "futures_theoretical": 182,
    "futures_theoretical_basis": 184,
    "futures_market_basis": 183,
    "futures_disparity_rate": 186,
    "futures_disparity": 185,
    "futures_open_interest": 195,
    "futures_open_interest_change": 181,
}

# FID 매핑 - 선물호가잔량
FID_FUTURES_QUOTE = {
    # 매도호가
    "futures_ask_1": 41, "futures_ask_2": 42, "futures_ask_3": 43, "futures_ask_4": 44, "futures_ask_5": 45,
    # 매도잔량
    "futures_ask_qty_1": 61, "futures_ask_qty_2": 62, "futures_ask_qty_3": 63, "futures_ask_qty_4": 64, "futures_ask_qty_5": 65,
    # 매도건수
    "futures_ask_cnt_1": 101, "futures_ask_cnt_2": 102, "futures_ask_cnt_3": 103, "futures_ask_cnt_4": 104, "futures_ask_cnt_5": 105,
    # 매수호가
    "futures_bid_1": 51, "futures_bid_2": 52, "futures_bid_3": 53, "futures_bid_4": 54, "futures_bid_5": 55,
    # 매수잔량
    "futures_bid_qty_1": 71, "futures_bid_qty_2": 72, "futures_bid_qty_3": 73, "futures_bid_qty_4": 74, "futures_bid_qty_5": 75,
    # 매수건수
    "futures_bid_cnt_1": 111, "futures_bid_cnt_2": 112, "futures_bid_cnt_3": 113, "futures_bid_cnt_4": 114, "futures_bid_cnt_5": 115,
    # 총잔량
    "futures_ask_total": 121,
    "futures_bid_total": 125,
}

# FID 매핑 - 주식체결 (기초자산)
FID_STOCK_TRADE = {
    "stock_time": 20,
    "stock_price": 10,
    "stock_change": 11,
    "stock_change_rate": 12,
    "stock_volume": 15,
    "stock_acc_volume": 13,
    "stock_open": 16,
    "stock_high": 17,
    "stock_low": 18,
}

# 전체 FID 통합 (실시간 등록용)
ALL_FIDS = set(FID_FUTURES_TRADE.values()) | set(FID_FUTURES_QUOTE.values()) | set(FID_STOCK_TRADE.values())

## 4. KiwoomFuturesCollector 클래스 정의

In [4]:
class KiwoomFuturesCollector:
    def __init__(self, futures_code, stock_code, screen_no, csv_path):
        self.futures_code = futures_code
        self.stock_code = stock_code
        self.screen_no = screen_no
        self.csv_path = csv_path
        
        self.app = QApplication(sys.argv)
        self.kiwoom = QAxWidget("KHOPENAPI.KHOpenAPICtrl.1")
        
        self.kiwoom.OnEventConnect.connect(self.on_event_connect)
        self.kiwoom.OnReceiveRealData.connect(self.on_receive_real_data)
        
        self.login_event_loop = QEventLoop()
        self.connected = False
        self.data_count = 0
        
        # 마지막 수신 데이터 캐시
        self.last_data = {col: "" for col in CSV_COLUMNS}
        self.last_data["futures_code"] = futures_code
        self.last_data["stock_code"] = stock_code
        
        # 수신된 모든 real_type 기록 (디버깅용)
        self.received_types = set()
        
        # CSV 헤더 작성
        self._init_csv()
    
    def _init_csv(self):
        """CSV 파일 초기화 (헤더 작성)"""
        df = pd.DataFrame(columns=CSV_COLUMNS)
        df.to_csv(self.csv_path, index=False, encoding="utf-8-sig")
        print(f"CSV 파일 생성: {self.csv_path}")
    
    def login(self):
        """로그인"""
        self.kiwoom.dynamicCall("CommConnect()")
        self.login_event_loop.exec_()
    
    def on_event_connect(self, err_code):
        """로그인 결과 처리"""
        if err_code == 0:
            print("\n[성공] 로그인 완료")
            self.connected = True
            
            user_name = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "USER_NAME")
            server_type = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "GetServerGubun")
            
            print(f"사용자: {user_name}")
            print(f"서버: {'모의투자' if server_type == '1' else '실투자'}")
        else:
            print(f"\n[실패] 로그인 실패 (에러코드: {err_code})")
            self.connected = False
        
        self.login_event_loop.exit()
    
    def set_real_reg(self):
        """실시간 등록 (선물 + 기초자산)"""
        fid_list = ";".join([str(fid) for fid in ALL_FIDS])
        
        # 선물 종목 등록
        result1 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.futures_code, fid_list, "0"
        )
        print(f"선물 실시간 등록 ({self.futures_code}): {result1}")
        
        # 기초자산(주식) 종목 추가 등록 (opt_type="1"로 추가)
        result2 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.stock_code, fid_list, "1"
        )
        print(f"주식 실시간 등록 ({self.stock_code}): {result2}")
        
        return result1, result2
    
    def get_real_data(self, code, fid):
        """실시간 데이터 조회"""
        value = self.kiwoom.dynamicCall(
            "GetCommRealData(QString, int)", code, fid
        ).strip()
        return value
    
    def _clean_value(self, value):
        """값 정리 (부호 제거)"""
        if value and (value.startswith("+") or value.startswith("-")):
            # 변화량은 부호 유지, 가격은 부호 제거
            if value[1:].replace(".", "").isdigit():
                return value[1:]
        return value
    
    def on_receive_real_data(self, code, real_type, real_data):
        """실시간 데이터 수신 및 CSV 저장"""
        
        # [디버깅] 새로운 real_type이 들어오면 출력
        if real_type not in self.received_types:
            self.received_types.add(real_type)
            print(f"\n>>> [NEW TYPE] real_type = '{real_type}' (code: {code}) <<<\n")
        
        # 처리할 실시간 타입
        valid_types = (
            "선물시세",       # 선물 체결/시세
            "선물호가잔량",   # 선물 호가
            "주식체결",       # 기초자산 체결
        )
        
        if real_type not in valid_types:
            return
        
        # 타입별로 FID 조회 및 캐시 업데이트
        if real_type == "선물시세" and code == self.futures_code:
            self.last_data["data_type"] = "futures_trade"
            for key, fid in FID_FUTURES_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
                    
        elif real_type == "선물호가잔량" and code == self.futures_code:
            self.last_data["data_type"] = "futures_quote"
            for key, fid in FID_FUTURES_QUOTE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
                    
        elif real_type == "주식체결" and code == self.stock_code:
            self.last_data["data_type"] = "stock_trade"
            for key, fid in FID_STOCK_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
        else:
            return
        
        # 타임스탬프 업데이트
        self.last_data["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        
        # CSV에 append
        self.data_count += 1
        df = pd.DataFrame([self.last_data])
        df.to_csv(self.csv_path, mode="a", header=False, index=False, encoding="utf-8-sig")
        
        # 콘솔 출력
        if real_type in ("선물시세", "선물호가잔량"):
            print(f"[{self.data_count}] {real_type:10} | "
                  f"선물: {self.last_data.get('futures_price', ''):>10} | "
                  f"이론가: {self.last_data.get('futures_theoretical', ''):>10} | "
                  f"베이시스: {self.last_data.get('futures_market_basis', ''):>8} | "
                  f"미결제: {self.last_data.get('futures_open_interest', ''):>8}")
        else:  # 주식체결
            print(f"[{self.data_count}] {real_type:10} | "
                  f"주식: {self.last_data.get('stock_price', ''):>10} | "
                  f"거래량: {self.last_data.get('stock_volume', ''):>10}")
    
    def run(self):
        """메인 실행"""
        # 종목코드 검증
        if not self.futures_code:
            print("[오류] FUTURES_CODE가 설정되지 않았습니다.")
            print("상단 설정 셀에서 선물 종목코드를 입력해주세요.")
            return
        
        self.login()
        
        if not self.connected:
            print("로그인 실패로 종료합니다.")
            return
        
        print("\n" + "=" * 60)
        print(f"주식선물 실시간 호가 수집 시작")
        print(f"  - 선물: {self.futures_code}")
        print(f"  - 기초자산: {self.stock_code}")
        print("=" * 60)
        
        self.set_real_reg()
        
        print("\n실시간 데이터 수집 중... (중지: Kernel Interrupt)")
        print(f"저장 위치: {self.csv_path}")
        print("\n수신 타입: 선물시세, 선물호가잔량, 주식체결")
        print("[디버깅] 새로운 real_type 수신 시 자동 출력됩니다.\n")
        
        self.app.exec_()
    
    def stop(self):
        """수집 중지"""
        print(f"\n수집 종료. 총 {self.data_count}개 데이터 저장됨.")
        print(f"저장 파일: {self.csv_path}")
        print(f"수신된 타입 목록: {self.received_types}")
        self.app.quit()

## 5. 실행

In [ ]:
# 수집기 생성 및 실행
collector = KiwoomFuturesCollector(
    futures_code=FUTURES_CODE,
    stock_code=STOCK_CODE,
    screen_no=SCREEN_NO,
    csv_path=CSV_PATH
)

# 실행 (중지하려면 Kernel > Interrupt 또는 Ctrl+C)
collector.run()

CSV 파일 생성: c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\option,futures MM\data\futures_A1164000_20260313_1051.csv

[성공] 로그인 완료
사용자: 박상혁
서버: 실투자

주식선물 실시간 호가 수집 시작
  - 선물: A1164000
  - 기초자산: 005930
선물 실시간 등록 (A1164000): 0
주식 실시간 등록 (005930): 0

실시간 데이터 수집 중... (중지: Kernel Interrupt)
저장 위치: c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\option,futures MM\data\futures_A1164000_20260313_1051.csv

수신 타입: 선물시세, 선물호가잔량, 주식체결
[디버깅] 새로운 real_type 수신 시 자동 출력됩니다.


>>> [NEW TYPE] real_type = '주식체결' (code: 005930) <<<

[1] 주식체결       | 주식:     184500 | 거래량:          5
[2] 주식체결       | 주식:     184500 | 거래량:          1

>>> [NEW TYPE] real_type = '주식선물호가잔량' (code: A1164000) <<<


>>> [NEW TYPE] real_type = '주식호가잔량' (code: 005930) <<<

[3] 주식체결       | 주식:     184500 | 거래량:         52

>>> [NEW TYPE] real_type = '선물시세' (code: A1164000) <<<

[4] 선물시세       | 선물:     184400 | 이론가:     184514 | 베이시스:      100 | 미결제:  3999206
[5] 선물시세       | 선물:     184400 | 이론가: 

KeyboardInterrupt: 

[589] 주식체결       | 주식:     184600 | 거래량:         56
[590] 주식체결       | 주식:     184500 | 거래량:         43
[591] 주식체결       | 주식:     184600 | 거래량:          9
[592] 주식체결       | 주식:     184500 | 거래량:       1376
[593] 주식체결       | 주식:     184500 | 거래량:          1
[594] 주식체결       | 주식:     184600 | 거래량:          1
[595] 주식체결       | 주식:     184600 | 거래량:          2
[596] 주식체결       | 주식:     184500 | 거래량:        313
[597] 주식체결       | 주식:     184500 | 거래량:          1
[598] 주식체결       | 주식:     184600 | 거래량:          3
[599] 주식체결       | 주식:     184500 | 거래량:         94
[600] 주식체결       | 주식:     184500 | 거래량:          1
[601] 주식체결       | 주식:     184500 | 거래량:        121
[602] 주식체결       | 주식:     184550 | 거래량:         10
[603] 주식체결       | 주식:     184600 | 거래량:        190
[604] 주식체결       | 주식:     184500 | 거래량:        112
[605] 주식체결       | 주식:     184600 | 거래량:          7
[606] 주식체결       | 주식:     184600 | 거래량:          6
[607] 주식체결       | 주식:     184600 | 거래량:          5
[608] 주식체결  

## 6. 수집된 데이터 확인

In [1]:
# 수집된 데이터 로드 및 미리보기
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
print(f"총 {len(df)}개 데이터")
df.head(10)

NameError: name 'pd' is not defined

In [ ]:
# 데이터 타입별 건수
df["data_type"].value_counts()

Series([], Name: count, dtype: int64)

In [ ]:
# 선물 시세 데이터만 필터링
df_futures = df[df["data_type"].isin(["futures_trade", "futures_quote"])]
print(f"선물 데이터: {len(df_futures)}건")
df_futures[["timestamp", "data_type", "futures_price", "futures_theoretical", 
            "futures_market_basis", "futures_open_interest"]].tail(10)

선물 데이터: 0건


,timestamp,data_type,futures_price,futures_theoretical,futures_market_basis,futures_open_interest


In [ ]:
# 기초자산(주식) 데이터만 필터링
df_stock = df[df["data_type"] == "stock_trade"]
print(f"주식 데이터: {len(df_stock)}건")
df_stock[["timestamp", "stock_price", "stock_volume", "stock_acc_volume"]].tail(10)

주식 데이터: 0건


,timestamp,stock_price,stock_volume,stock_acc_volume
